# Auditing the Farm Survey

A working answer is not the same as a usable one. When a survey grows from 5,654 fields toward a million, the gap between `O(n)` and `O(log n)` is the gap between an inspector getting her reading and missing her window. In this project you build an **auditing engine** for the Maji Ndogo farm survey: linear and binary search to find a field, a recursive merge sort to rank the harvest, lambda-powered `map` and `filter` to flag and project, and the Big O vocabulary to say precisely why each choice scales. It is exactly the kind of efficiency-minded work on real data that belongs in a data-science portfolio.

The radio crackles at 2:14 p.m.

"Field 672. Pollution level. Now."

The inspector is standing in the middle of a potato field in the Sokoto lowlands with a clipboard and a deadline. Her truck's satellite link is down. She needs the pollution reading for that specific field — out of 5,654 records — in under ten seconds, or she misses her reporting window and the entire audit gets rescheduled.

On your laptop, you have the farm survey database. You have Python. You have a list of 5,654 dictionaries.

You also have a problem: the code you have right now loops through every single record from the first to the last, checking each one. On this dataset, that takes less than a second. But Maji Ndogo's ambition is to grow this system to cover one million farms across the region within five years. At that scale, the same loop takes minutes. The inspector does not have minutes.

**This is the challenge of algorithmic efficiency.** The question is not just "does this code work?" — it is "does this code work *fast enough* when the data is ten times larger? A hundred times larger? A million records?" The Big O notation you are about to learn is the vocabulary for answering that question precisely.

In [32]:

import sqlite3

# Connect to the Maji Ndogo farm survey database.
# Make sure the .db file is in the same folder as this notebook.
connection = sqlite3.connect('Maji_Ndogo_farm_survey_small.db')
cursor = connection.cursor()

cursor.execute(
    """SELECT Field_ID, Pollution_level, Plot_size, Annual_yield, Crop_type, Standard_yield
       FROM farm_management_features"""
)

# The Crop_type and Annual_yield labels were swapped when the survey was digitised,
# so we swap them back while building our list of field records.
field_records = []
for row in cursor.fetchall():
    field_records.append({
        'Field_ID': row[0],
        'Pollution_level': round(row[1], 4),
        'Plot_size': row[2],
        'Crop_type': row[3],                # stored under 'Annual_yield' in the database
        'Annual_yield': round(row[4], 4),   # stored under 'Crop_type' in the database
        'Standard_yield': round(row[5], 4),
    })

connection.close()

print(f"Loaded {len(field_records)} field records.")

Loaded 5654 field records.


In [33]:

maji_ndogo_sample = [
    {'Field_ID': 40734, 'Pollution_level': 0.0853, 'Plot_size': 1.3, 'Crop_type': 'cassava', 'Annual_yield': 0.7514, 'Standard_yield': 0.578},
    {'Field_ID': 30629, 'Pollution_level': 0.3997, 'Plot_size': 2.2, 'Crop_type': 'cassava', 'Annual_yield': 1.0699, 'Standard_yield': 0.4863},
    {'Field_ID': 39924, 'Pollution_level': 0.358, 'Plot_size': 3.4, 'Crop_type': 'tea', 'Annual_yield': 2.2088, 'Standard_yield': 0.6496},
    {'Field_ID': 5754, 'Pollution_level': 0.2867, 'Plot_size': 2.4, 'Crop_type': 'cassava', 'Annual_yield': 1.2776, 'Standard_yield': 0.5323},
    {'Field_ID': 14146, 'Pollution_level': 0.0432, 'Plot_size': 1.5, 'Crop_type': 'wheat', 'Annual_yield': 0.8326, 'Standard_yield': 0.5551}
]

## Challenge 1.1: Linear search

The survey records are stored in the order the teams submitted them — no sorting, no index. When an inspector calls in a `Field_ID`, we start at record one and check every record until we find a match. This is a **linear search**: `O(n)` — in the worst case, it inspects every record.

### Task

Complete `linear_search_field(field_list, target_id)`. It must:
- Iterate through `field_list`.
- Return the `Pollution_level` of the first record whose `Field_ID` matches `target_id`.
- Return `None` if no match is found.

> ⚠️ Do not change the function name `linear_search_field`.

In [34]:

def linear_search_field(field_list, target_id):
    for record in field_list:
        if record.get('Field_ID') == target_id:
            return record.get('Pollution_level')
    return None


In [35]:
linear_search_field(maji_ndogo_sample, 39924)

0.358

In [36]:
print(linear_search_field(maji_ndogo_sample, 99999))

None


In [37]:

linear_search_field(field_records, 672)

0.6131

## Challenge 2.1: Binary search

If the records are **sorted by `Field_ID`**, we can do much better. Jump to the middle record. Is the target ID higher or lower? Discard half the list. Repeat. This is a **binary search**: `O(log n)` — at one million records, it takes roughly 20 comparisons instead of one million.

The cost: the data must be sorted first.

### Task

Complete `binary_search_yield(sorted_field_list, target_id)`. Assume the input is already sorted by `Field_ID` in ascending order. Return the `Annual_yield` of the matching record, or `None` if not found.

> ⚠️ Do not change the function name `binary_search_yield`.

In [38]:
### START FUNCTION
def binary_search_yield(sorted_field_list, target_id):
    left = 0
    right = len(sorted_field_list) - 1

    while left <= right:
        mid = (left + right) // 2
        record = sorted_field_list[mid]

        if record.get('Field_ID') == target_id:
            return record.get('Annual_yield')
        elif record.get('Field_ID') < target_id:
            left = mid + 1
        else:
            right = mid - 1
    return None
### END FUNCTION

In [39]:
sorted_sample = sorted(maji_ndogo_sample, key=lambda f: f['Field_ID'])
binary_search_yield(sorted_sample, 30629)

1.0699

In [40]:
_sorted_sample = sorted(maji_ndogo_sample, key=lambda f: f['Field_ID'])
print(binary_search_yield(_sorted_sample, 11111))

None


## Challenge 3.1: Merge sort

Binary search needs sorted data. We need a sorter. Bubble sort is `O(n²)`: double the records, quadruple the work. **Merge sort** is `O(n log n)`: it splits the list in half, sorts each half **recursively**, then merges the two sorted halves.

Pseudocode:
```
merge_sort(list):
    if len(list) <= 1: return list
    mid   = len(list) // 2
    left  = merge_sort(list[:mid])
    right = merge_sort(list[mid:])
    return merge(left, right)
```

### Task

Complete `merge_sort_yields(field_list)` and the helper `merge(left, right)`. Sort records into **descending** order of `Annual_yield` (highest yield first). `merge_sort_yields` must be recursive and must call `merge`.

> ⚠️ Do not change either function name.

In [41]:

def merge_sort_yields(field_list):
    if len(field_list) <= 1:
        return field_list
    
    mid = len(field_list) // 2
    left = merge_sort_yields(field_list[:mid])
    right = merge_sort_yields(field_list[mid:])

    return merge(left, right)

def merge(left, right):
    sorted_list = []
    i = 0
    j = 0

    while i < len(left) and j < len(right):
        if left[i].get('Annual_yield', 0) <= right[j].get('Annual_Yield', 0):
            sorted_list.append(left[i])
            i += 1
        else:
            sorted_list.append(right[j])
            j += 1


    sorted_list.extend(left[i:])
    sorted_list.extend(right[j:])

    return sorted_list

In [42]:
merge_sort_yields(maji_ndogo_sample)

[{'Field_ID': 14146,
  'Pollution_level': 0.0432,
  'Plot_size': 1.5,
  'Crop_type': 'wheat',
  'Annual_yield': 0.8326,
  'Standard_yield': 0.5551},
 {'Field_ID': 5754,
  'Pollution_level': 0.2867,
  'Plot_size': 2.4,
  'Crop_type': 'cassava',
  'Annual_yield': 1.2776,
  'Standard_yield': 0.5323},
 {'Field_ID': 39924,
  'Pollution_level': 0.358,
  'Plot_size': 3.4,
  'Crop_type': 'tea',
  'Annual_yield': 2.2088,
  'Standard_yield': 0.6496},
 {'Field_ID': 30629,
  'Pollution_level': 0.3997,
  'Plot_size': 2.2,
  'Crop_type': 'cassava',
  'Annual_yield': 1.0699,
  'Standard_yield': 0.4863},
 {'Field_ID': 40734,
  'Pollution_level': 0.0853,
  'Plot_size': 1.3,
  'Crop_type': 'cassava',
  'Annual_yield': 0.7514,
  'Standard_yield': 0.578}]

In [43]:
[f['Annual_yield'] for f in merge_sort_yields(maji_ndogo_sample)[:3]]

[0.8326, 1.2776, 2.2088]

## Challenge 4.1: Lambda and filter — flagging polluted fields

Any field above a pollution threshold needs an inspection visit, and the list must update as thresholds change. Python's `filter()` with a `lambda` makes this one readable line.

### Task

Complete `identify_high_pollution(field_list, threshold)`. Use `filter()` and a `lambda` to return only the records whose `Pollution_level` is **strictly greater than** `threshold`. Return the result as a list.

> ⚠️ Do not change the function name `identify_high_pollution`.

In [44]:

def identify_high_pollution(field_list, threshold):
    
    return list(filter(lambda record: record.get('Pollution_level', 0) > threshold, field_list))


In [45]:
identify_high_pollution(maji_ndogo_sample, 0.3)

[{'Field_ID': 30629,
  'Pollution_level': 0.3997,
  'Plot_size': 2.2,
  'Crop_type': 'cassava',
  'Annual_yield': 1.0699,
  'Standard_yield': 0.4863},
 {'Field_ID': 39924,
  'Pollution_level': 0.358,
  'Plot_size': 3.4,
  'Crop_type': 'tea',
  'Annual_yield': 2.2088,
  'Standard_yield': 0.6496}]

## Challenge 5.1: Lambda and map — projecting harvests

If `filter()` decides which items to *keep*, `map()` transforms *every* item. The Ministry of Agriculture wants a quick harvest projection for each field: `Plot_size` × `Standard_yield`, rounded to 2 decimal places.

### Task

Complete `project_harvests(field_list)`. Use `map()` and a `lambda` to compute each field's projected yield (`Plot_size * Standard_yield`, rounded to 2 decimals). Return the result as a list.

> ⚠️ Do not change the function name `project_harvests`.

In [46]:

def project_harvests(field_list):
    return list(map(
            lambda record: round(
                record.get('Plot_size', 0) * record.get('Standard_yield', 0),2),field_list))


In [47]:
project_harvests(maji_ndogo_sample[2:4])

[2.21, 1.28]

In [48]:
project_harvests(maji_ndogo_sample)

[0.75, 1.07, 2.21, 1.28, 0.83]

## PART TWO 

# Modeling the Field Registry

A dictionary will let you set a field's pollution level to 7,000 and never complain. That is fine for a throwaway script and dangerous for a system five engineers will build on. As international consultants assisting Maji Ndogo's agricultural registry modernization initiative, you will retire the loose dictionaries and model the survey data using robust object-oriented programming (OOP). You will design a base `Field` class that validates its own data, crop-specific subclasses that price their own harvest, an abstract base that prevents structural inconsistencies, and a unified `FieldRegistry` class that aggregates vital metrics across thousands of records cleanly. Encapsulation, inheritance, polymorphism, and abstraction will be applied directly to real survey data using professional standards. A genuine asset to demonstrate enterprise data practices.

The Auditing Engine is running and the search functions are fast. But Sanaa calls you into her office with a whiteboard already covered in red marker.

"Here's the problem," she says, drawing a box labeled `dict`. "Every field record is a dictionary. Nothing stops anyone from setting `Pollution_level` to 7,000. Nothing ensures `Plot_size` is always there. Nothing groups the behavior that belongs with the data." She draws a line through the box. "A dictionary is fine for one-off scripts. We are building a system that five other engineers will build on top of. Loose data breaks systems."

She turns to you. "We need to redesign."

This is where **object-oriented programming** comes in. Instead of passing dictionaries around and hoping everyone handles them consistently, we define a class — a blueprint that declares what every field *has* (attributes) and what every field *can do* (methods). The blueprint enforces structure. Other code can trust it.

By the end of this session you will have built the **Maji Ndogo Field Registry**: a set of Python classes that model the survey data with proper validation, inheritance, and polymorphism — the kind of code that professional engineering teams actually build on.

## Challenge 1.2: The `Field` class

A class is a blueprint. Define it once, and every field in the survey can become an object.

Sensors also malfunction. A reading of `7.5` on a 0–1 scale is physically impossible. **Encapsulation** lets a class control its own data: a `@property` with a setter intercepts every write and enforces the valid range. A field's identity and its self-protection belong to the same blueprint, so you build both here.

### Task

Complete the class `Field`:
- `__init__` must accept `field_id`, `plot_size`, `standard_yield`, and `pollution_level` (**defaulting to `0.0`** — a field with no reading is treated as clean), storing each as an instance attribute with the same PEP 8 `snake_case` name.
- `describe()` must **return** (not print) the string: `'Field {field_id}: {plot_size} Ha (standard yield {standard_yield})'`
- `calculate_yield()` must **return** `round(plot_size * standard_yield, 2)`.
- Back `pollution_level` with a `@property` and a private attribute `_pollution_level`. The setter must raise a `ValueError` with the message `'Pollution level must be between 0 and 1.'` if the value falls outside `[0, 1]`.
- Add a class docstring and a docstring for each method.

> ⚠️ Do not change the class or method names.

In [49]:
class Field:
    """Represents a surveyed field with validated pollution monitoring."""

    def __init__(self, field_id, plot_size, standard_yield, pollution_level=0.0):
        """Initialize the field and set its pollution level."""
        self.field_id = field_id
        self.plot_size = plot_size
        self.standard_yield = standard_yield
        self.pollution_level = pollution_level

    @property
    def pollution_level(self):
        """Return the field's pollution level."""
        return self._pollution_level

    @pollution_level.setter
    def pollution_level(self, value):
        """Set the pollution level if it is within the valid range."""
        if not 0 <= value <= 1:
            raise ValueError("Pollution level must be between 0 and 1.")
        self._pollution_level = value

    def describe(self):
        """Return a description of the field."""
        return (
            f"Field {self.field_id}: {self.plot_size} Ha "
            f"(standard yield {self.standard_yield})"
            )

    def calculate_yield(self):
        """Return the calculated and rounded field yield."""
        return round(self.plot_size * self.standard_yield, 2)


In [50]:
f = Field(39924, 3.4, 0.65)
f.describe()

'Field 39924: 3.4 Ha (standard yield 0.65)'

In [51]:
try:
    Field(40734, 1.3, 0.58, 7.5)
except ValueError as e:
    print(e)

Pollution level must be between 0 and 1.


## Challenge 2: Inheritance — crop-specific subclasses

Tea, coffee, and wheat each sell at a different price per ton. **Inheritance** lets us create specialized subclasses that extend `Field` with crop-specific behavior without rewriting the shared logic. Because each subclass implements the same method names, code can call `describe()` or `projected_revenue()` on any of them without knowing which crop it is — that is **polymorphism**.

### Task

Create three subclasses of `Field`: `TeaField` ($1,180/ton), `CoffeeField` ($2,400/ton), `WheatField` ($320/ton). Each must:
- Store the price as a class attribute `PRICE_PER_TON`.
- Override `describe()` to prefix with the crop name (e.g. `'Tea field 39924: ...'`).
- Add a method `projected_revenue()` returning `round(calculate_yield() * PRICE_PER_TON, 2)`.
- Include a class docstring.

> ⚠️ Do not change the class or method names.

In [52]:
class TeaField(Field):
    """Tea crop field."""
    PRICE_PER_TON = 1180

    def describe(self):
        return (
        f"Tea field {self.field_id}: {self.plot_size} Ha "
        f"(standard yield {self.standard_yield})"
        )

    def projected_revenue(self):
        return round(self.calculate_yield() * self.PRICE_PER_TON, 2)


class CoffeeField(Field):
    """Coffee crop field."""
    PRICE_PER_TON = 2400

    def describe(self):
        return (
        f"Coffee field {self.field_id}: {self.plot_size} Ha "
        f"(standard yield {self.standard_yield})"
        )

    def projected_revenue(self):
        return round(self.calculate_yield() * self.PRICE_PER_TON, 2)


class WheatField(Field):
    """Wheat crop field."""
    PRICE_PER_TON = 320

    def describe(self):
        return (
        f"Wheat field {self.field_id}: {self.plot_size} Ha "
        f"(standard yield {self.standard_yield})"
                )

    def projected_revenue(self):
        return round(self.calculate_yield() * self.PRICE_PER_TON, 2)

In [53]:
TeaField(39924, 3.4, 0.65, 0.36).describe()

'Tea field 39924: 3.4 Ha (standard yield 0.65)'

In [54]:
TeaField(39924, 3.4, 0.65, 0.36).projected_revenue()

2607.8

## Challenge 3: Abstraction — an abstract base class

Nothing prevents a developer from creating a plain `AbstractCropField` object when they should have used a concrete subclass. **Abstraction** via an abstract base class (ABC) makes that impossible — the class cannot be instantiated until its abstract methods are implemented.

### Task

Create `AbstractCropField(ABC)` with:
- `__init__` accepting `field_id`, `plot_size`, and `standard_yield`.
- An abstract method `crop_name()` — every concrete subclass must implement it.
- `calculate_yield()` returning `round(plot_size * standard_yield, 2)`.
- A class docstring.

Then create `MaizeField(AbstractCropField)` implementing `crop_name()` to return `'maize'`.

> ⚠️ Do not change the class or method names.

In [55]:
### START FUNCTION
from abc import ABC, abstractmethod

class AbstractCropField(ABC):
    """Abstract base - every concrete crop field must implement crop_name()."""

    def __init__(self, field_id, plot_size, standard_yield):
        self.field_id = field_id
        self.plot_size = plot_size
        self.standard_yield = standard_yield

    @abstractmethod
    def crop_name(self):
        pass
    def calculate_yield(self):
        
        return round(self.plot_size * self.standard_yield, 2)

class MaizeField(AbstractCropField):
    """Concrete maize field."""

    def crop_name(self):
        # Insert your code here
        return 'maize'
### END FUNCTION

In [56]:
try:
    AbstractCropField(41964, 4.1, 0.55)
except TypeError as e:
    print(e)

Can't instantiate abstract class AbstractCropField without an implementation for abstract method 'crop_name'


In [57]:
MaizeField(41964, 4.1, 0.55).crop_name()

'maize'

## Challenge 4.2: The `build_field` factory

The survey hands you raw dictionaries. Something has to decide which class each record becomes, and that decision belongs in one place rather than scattered across every caller. A **factory function** takes a record and hands back the right object.

### Task

Complete `build_field(record)`:
- Accept a raw survey record dictionary.
- Return the correct `TeaField`, `CoffeeField`, or `WheatField` object based on `Crop_type` (case-insensitive).
- Return `None` for any crop not in the pricing catalog.
- Include a docstring.

> ⚠️ Do not change the function name.

In [58]:

def build_field(record):
    crop_type = record.get("Crop_type", "").strip().lower()

    if crop_type == 'tea':
        return TeaField(
            field_id=record.get('Field_id'),
            plot_size=record.get('plot_size'),
            standard_yield=record.get('standard_yield')
        )
    elif crop_type == 'tea':
            return CoffeeField(
                field_id=record.get('Field_id'),
                plot_size=record.get('plot_size'),
                standard_yield=record.get('standard_yield')
            )
    elif crop_type == 'tea':
            return WheatField(
                field_id=record.get('Field_id'),
                plot_size=record.get('plot_size'),
                standard_yield=record.get('standard_yield')
            )
    return None


In [59]:
# input 1
tea_record = {'Field_ID': 39924, 'Plot_size': 3.4, 'Standard_yield': 0.65,
              'Crop_type': 'tea', 'Pollution_level': 0.36}
type(build_field(tea_record)).__name__

'TeaField'

In [60]:
# input 2
cassava_record = {'Field_ID': 41964, 'Plot_size': 4.1, 'Standard_yield': 0.55,
                  'Crop_type': 'cassava', 'Pollution_level': 0.2}
build_field(cassava_record)

## Challenge 5.2: The `FieldRegistry`

Now bring it together. The registry aggregates every priced field, calling the same method names on each one regardless of crop. That shared interface is polymorphism doing the real work.

### Task

Complete the `FieldRegistry` class:
- Accept a list of raw records in `__init__`, use `build_field()` to build `self.fields` (excluding `None` values).
- `size()` → number of priced fields.
- `total_projected_yield()` → sum of all `calculate_yield()` values, rounded to 2 dp.
- `total_projected_revenue()` → sum of all `projected_revenue()` values, rounded to 2 dp.
- Include a class docstring and method docstrings.

> ⚠️ Do not change the class or method names.

In [61]:


class FieldRegistry:
    """Registry of priced crop-field objects built from the survey dataset."""

    def __init__(self, records):
        """Build and store all supported crop fields."""
        self.field_data = []

        for record in records:
            field = build_field(record=record)
            if field is not None:
                self.field_data.append(field)

    def size(self):
        """Return the number of priced fields."""
        return len(self.field_data)

    def total_projected_yield(self):
        """Return the sum of projected yields, rounded to two decimals."""
        return round(sum(field.calculate_yield() for field in self.field_data),
                2)

    def total_projected_revenue(self):
        """Return the sum of projected revenues, rounded to two decimals."""
        return round(sum(field.projected_revenue() for field in self.field_data),2)
        


## PART 3

# Understanding Maji Ndogo's Agriculture

The Minister wants a planting recommendation by Friday morning, and the answer is sitting in a database nobody can question fast enough. This week you turn four raw survey tables into a single **Pandas** DataFrame and interrogate it: which crop thrives in the highlands, which soils are most productive, and exactly which fields meet every condition for a strong harvest. Loading and joining SQL tables, cleaning the messy real-world entries, then grouping, filtering, sorting, and querying your way to an answer — the core data-analysis loop, on real data, delivered against a deadline. The closing piece of your Maji Ndogo engagement, and the one the Minister will actually read.

Sanaa's phone rings on Thursday afternoon.

"The Minister needs a crop recommendation for the northern highland plots by Friday morning," the assistant says. "Specifically: which crop performs best up there, what soil types are most productive, and which fields should be prioritized for the next planting cycle."

Sanaa puts the phone down and looks at you. "We have the data. We have the Field Registry. What we do not have is a fast way to ask it questions."

That changes today. The tools you need are already installed: **Pandas** turns the farm survey database into a structured DataFrame that you can filter, group, sort, and transform with single lines of code. The analysis that takes an afternoon with raw Python takes twenty minutes with Pandas.

The Ministry is waiting.

In [ ]:
### DO NOT CHANGE ANYTHING IN THIS CELL, ONLY EXECUTE IT!

import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine('sqlite:///Maji_Ndogo_farm_survey_small.db')

sql_query = """
SELECT *
FROM geographic_features
LEFT JOIN weather_features USING (Field_ID)
LEFT JOIN soil_and_crop_features USING (Field_ID)
LEFT JOIN farm_management_features USING (Field_ID)
"""

with engine.connect() as connection:
    MD_agric_df = pd.read_sql_query(text(sql_query), connection)

# Wrangling: the 'Crop_type' and 'Annual_yield' labels were swapped on export, so swap them back;
# elevations were stored with the wrong sign; and a few crop names were misspelt.
MD_agric_df.rename(columns={'Annual_yield': 'Crop_type_Temp', 'Crop_type': 'Annual_yield'}, inplace=True)
MD_agric_df.rename(columns={'Crop_type_Temp': 'Crop_type'}, inplace=True)
MD_agric_df['Elevation'] = MD_agric_df['Elevation'].abs()
corrections = {'cassaval': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'}
MD_agric_df['Crop_type'] = MD_agric_df['Crop_type'].apply(
    lambda crop: corrections.get(crop.strip(), crop.strip())
)
print(f'Loaded: {MD_agric_df.shape[0]} rows × {MD_agric_df.shape[1]} columns')

Loaded: 5654 rows × 18 columns


### Data dictionary

**Geographic features**
| Column | Description | Type |
|--------|-------------|------|
| Field_ID | Unique identifier for each field | BigInt |
| Elevation | Elevation above sea level in meters | Float |
| Latitude / Longitude | Geographic coordinates in degrees | Float |
| Location | Province the field is in | Text |
| Slope | Slope of the land in degrees | Float |

**Weather features**
| Column | Description | Type |
|--------|-------------|------|
| Rainfall | Annual rainfall in mm | Float |
| Min_temperature_C | Average minimum temperature in °C | Float |
| Max_temperature_C | Average maximum temperature in °C | Float |
| Ave_temps | Average temperature in °C — mean of Min and Max | Float |

**Soil features**
| Column | Description | Type |
|--------|-------------|------|
| Soil_fertility | Normalized fertility index, 0–1 | Float |
| Soil_type | Categorical soil classification | Text |
| pH | Soil pH level | Float |

**Farm management features**
| Column | Description | Type |
|--------|-------------|------|
| Pollution_level | Normalized pollution index, 0–1 | Float |
| Plot_size | Field area in hectares | Float |
| Crop_type | Type of crop grown | Text |
| Annual_yield | Total annual yield in tonnes | Float |
| Standard_yield | Normalized yield score, 0–1 | Float |

## Challenge 1.3: Crop distribution

Before recommending where to plant, we need to understand where crops currently grow and what conditions they share. For a given crop type, the average rainfall and average elevation immediately tells us whether it is a highland or lowland species.

### Task

Complete `explore_crop_distribution(df, crop_filter)`. It must filter the DataFrame to rows where `Crop_type` matches `crop_filter`, then return a **tuple** of `(mean Rainfall, mean Elevation)` as plain Python floats.

> ⚠️ Do not change the function name `explore_crop_distribution`.

In [ ]:
### START FUNCTION
def explore_crop_distribution(df, crop_filter):
    # Insert your code here
    filtered_df = df[df['Crop_type'] == crop_filter]
    distribution = filtered_df['Rainfall'].mean(), filtered_df['Elevation'].mean()

    return distribution
### END FUNCTION

In [ ]:
explore_crop_distribution(MD_agric_df, 'tea')

(np.float64(1534.5079956188388), np.float64(775.208667535597))

In [ ]:
explore_crop_distribution(MD_agric_df, 'wheat')

(np.float64(1010.2859910581222), np.float64(595.8384148002981))

## Challenge 2.3: Soil fertility by type

Soil type is one of the strongest predictors of crop success.

### Task

Complete `analyse_soil_fertility(df)`. It must group the DataFrame by `Soil_type`, calculate the mean `Soil_fertility` per group, and return the result as a **Pandas Series**.

> ⚠️ Do not change the function name `analyse_soil_fertility`.

### Expected output

```
Soil_type
Loamy       0.585868
Peaty       0.604882
Rocky       0.582368
Sandy       0.595669
Silt        0.652654
Volcanic    0.648894
Name: Soil_fertility, dtype: float64
```

In [ ]:

def analyse_soil_fertility(df):
    return df.groupby('Soil_type')['Soil_fertility'].mean()


In [ ]:
analyse_soil_fertility(MD_agric_df)

Soil_type
Loamy       0.585868
Peaty       0.604882
Rocky       0.582368
Sandy       0.595669
Silt        0.652654
Volcanic    0.648894
Name: Soil_fertility, dtype: float64

analyse_soil_fertility(MD_agric_df)

## Challenge 3: Climate and geography by crop

For every crop type, we need the average elevation, minimum temperature, maximum temperature, and rainfall.

### Task

Complete `climate_geography_influence(df, column)`. It must group the DataFrame by `column`, calculate the mean of `Elevation`, `Min_temperature_C`, `Max_temperature_C`, and `Rainfall` — **in that order** — and return the result as a **DataFrame**.

> ⚠️ Do not change the function name `climate_geography_influence`.

### Expected output

```
             Elevation  Min_temperature_C  Max_temperature_C    Rainfall
Crop_type
banana      487.973572          -5.354344          31.988152  1659.905687
cassava     682.903008          -3.992113          30.902381  1210.543006
coffee      647.047734          -4.028007          30.855189  1527.265074
maize       680.596982          -4.497995          30.576692   681.010276
potato      696.313917          -4.375334          30.300608   660.289064
rice        352.858053          -6.610566          32.727170  1632.382642
tea         775.208668          -2.862651          29.950383  1534.507996
wheat       595.838415          -4.968107          30.973845  1010.285991
```

In [ ]:

def climate_geography_influence(df, column):
    metrics = ['Elevation', 'Min_temperature_C', 'Max_temperature_C', 'Rainfall']
    return df.groupby(column)[metrics].mean()

In [ ]:
climate_geography_influence(MD_agric_df, 'Crop_type')

,Elevation,Min_temperature_C,Max_temperature_C,Rainfall
Crop_type,,,,
banana,487.973572,-5.354344,31.988152,1659.905687
cassava,682.903008,-3.992113,30.902381,1210.543006
coffee,647.047734,-4.028007,30.855189,1527.265074
maize,680.596982,-4.497995,30.576692,681.010276
potato,696.313917,-4.375334,30.300608,660.289064
rice,352.858053,-6.610566,32.727170,1632.382642
tea,775.208668,-2.862651,29.950383,1534.507996
wheat,595.838415,-4.968107,30.973845,1010.285991


## Challenge 4.3: Top-performing crop

Time to identify Maji Ndogo's single best-performing crop — the one with the most fields exceeding the average `Standard_yield`.

**Hint:** After grouping, the labels of the grouping column can be accessed with `.index`. For example:
```python
grouped_df = MD_agric_df.groupby('Soil_type').mean(numeric_only=True).sort_values('Elevation', ascending=False)
print(grouped_df.index[0])
```

### Task

Complete `find_ideal_fields(df)`. It must:
1. Filter to fields with an above-average `Standard_yield`.
2. Group by `Crop_type` and count records per group.
3. Sort to put the highest count first.
4. Return the `Crop_type` name at index 0 as a **string**.

> ⚠️ Do not change the function name `find_ideal_fields`.

### Expected output

**Input 1:** `type(find_ideal_fields(MD_agric_df))` → `<class 'str'>`

**Input 2:** `print('Top-performing crop:', find_ideal_fields(MD_agric_df))` → `Top-performing crop: tea`

In [ ]:

def find_ideal_fields(df):
    filtered_df = df[(df['Standard_yield'] > df['Standard_yield'].mean())]
    return str(filtered_df.groupby('Crop_type')['Field_ID'].count().sort_values(ascending=False).index[0])

In [ ]:
type(find_ideal_fields(MD_agric_df))

str

In [ ]:
print('Top-performing crop:', find_ideal_fields(MD_agric_df))

Top-performing crop: tea


## Challenge 5: Ideal growing conditions

Given a crop type, return only the fields that meet all four quality criteria simultaneously. This is a natural fit for the `.query()` method, which reads almost like the sentence above — though a boolean mask works just as well.

### Task

Complete `find_good_conditions(df, crop_type)`. Filter to rows where:
1. `Crop_type` matches `crop_type`.
2. `Standard_yield` is above that crop's own mean `Standard_yield`.
3. `Ave_temps` is between 12 and 15 (inclusive).
4. `Pollution_level` is below 0.0001.

Return the filtered **DataFrame**.

> 📌 `Ave_temps` is pre-computed in the database as the average of `Min_temperature_C` and `Max_temperature_C`.

> ⚠️ Do not change the function name `find_good_conditions`.

In [ ]:
### START FUNCTION
def find_good_conditions(df, crop_type):
    filtered_df = df[df['Crop_type'] == crop_type]
    good_conditions = filtered_df[
        (filtered_df['Standard_yield'] > filtered_df['Standard_yield'].mean()) &
        (filtered_df['Ave_temps'] > 12) & (filtered_df['Ave_temps'] < 15) &
        (filtered_df['Pollution_level'] < 0.0001)
    ]
    return good_conditions
### END FUNCTION

In [ ]:
find_good_conditions(MD_agric_df, 'tea').shape

(14, 18)

In [ ]:
find_good_conditions(MD_agric_df, find_ideal_fields(MD_agric_df)).head()

,Field_ID,Elevation,Latitude,Longitude,Location,Slope,Rainfall,Min_temperature_C,Max_temperature_C,Ave_temps,Soil_fertility,Soil_type,pH,Pollution_level,Plot_size,Annual_yield,Crop_type,Standard_yield
197,16235,688.63477,-14.585503,-5.948055,Rural_Hawassa,2.506081,1569.9,-3.6,31.6,14.00,0.62,Volcanic,4.117729,1.007224e-08,10.4,7.484284,tea,0.719643
852,16330,739.42730,-14.405275,-6.267883,Rural_Hawassa,7.132496,1687.9,-3.1,31.1,14.00,0.64,Volcanic,4.538485,7.108529e-09,2.1,1.664479,tea,0.792609
869,27813,691.24520,-14.400770,-5.966074,Rural_Hawassa,11.563782,1640.5,-3.5,32.7,14.60,0.65,Volcanic,4.123869,1.691890e-08,3.0,2.219570,tea,0.739857
943,13657,713.74615,-14.549458,-6.051661,Rural_Hawassa,14.432365,1604.0,-3.4,27.7,12.15,0.66,Volcanic,5.147911,7.575035e-09,0.7,0.554950,tea,0.792785
1293,18498,648.13605,-14.297139,-6.146258,Rural_Hawassa,9.684363,1648.5,-3.8,29.5,12.85,0.65,Volcanic,5.154659,1.387864e-08,2.5,2.095446,tea,0.838179
